# Notebook 3 — Modelagem: Elastic Net e Classificacao Binaria

Neste notebook construimos os modelos de machine learning do artigo.
Partindo das features do Notebook 2, treinamos:

1. **Modelo de regressao** (Elastic Net) — preve a vida util em ciclos
2. **Classificador binario** (Regressao Logistica) — separa baterias em
   vida longa vs vida curta usando apenas os **5 primeiros ciclos**

O modelo usa apenas **41 celulas de treino** e atinge ~9% de erro percentual.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from sklearn.linear_model import LinearRegression, ElasticNetCV, LogisticRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3})
print('Bibliotecas carregadas.')

def simular_curva_bruta(ciclo, taxa_deg=0.001, semente=42):
    rng = np.random.default_rng(semente + ciclo * 7)
    n_pts = 190 + rng.integers(-25, 40); deg = taxa_deg * ciclo
    V = np.clip(np.linspace(3.5, 2.0, n_pts) + rng.normal(0, 0.003, n_pts), 2.0, 3.5)
    V = np.sort(V)[::-1]; x = (3.5 - V) / 1.5
    p1 = 1/(1+np.exp(-28*(x-0.12-deg*0.25))); p2 = 1/(1+np.exp(-18*(x-0.62-deg*0.4)))
    p3 = 1/(1+np.exp(-12*(x-0.92)))
    Q = 1.1*(1-0.08*deg)*(0.42*p1+0.46*p2+0.12*p3) + rng.normal(0,0.0015,n_pts)
    return V, np.clip(Q, 0, None)

def padronizar_curva(V_bruto, Q_bruto, n_pontos=1000):
    V_grid = np.linspace(3.5, 2.0, n_pontos)
    V_ord = V_bruto[::-1].copy(); Q_ord = Q_bruto[::-1].copy()
    _, idx = np.unique(V_ord, return_index=True)
    return V_grid, UnivariateSpline(V_ord[idx], Q_ord[idx], s=0, ext='const')(V_grid)


## 1. Construindo o Dataset de Features

Geramos **124 celulas** (mesmo tamanho do artigo) e extraimos 6 features por celula:

| Feature | Fonte | Descricao |
|---------|-------|-----------|
| `log_var_dQ` | Delta Q_{100-10}(V) | **Preditor dominante** — variancia em log |
| `min_dQ` | Delta Q_{100-10}(V) | Minimo da curva diferencial |
| `mean_dQ` | Delta Q_{100-10}(V) | Media da curva diferencial |
| `slope_Q` | Q ciclos 2-100 | Taxa de queda da capacidade |
| `Q_ciclo2` | Ciclo 2 | Capacidade inicial (baixa correlacao com vida util!) |
| `Q_ciclo100` | Ciclo 100 | Capacidade no ciclo 100 |


In [ ]:
np.random.seed(42)
N = 124
taxas = np.random.uniform(0.0003, 0.0065, N)
vidas = np.clip((0.080/taxas)*np.exp(np.random.normal(0,0.15,N)), 150, 2300).astype(int)

print('Extraindo features das 124 celulas...', end='')
features = []
nomes = ['log_var_dQ', 'min_dQ', 'mean_dQ', 'slope_Q', 'Q_ciclo2', 'Q_ciclo100']

for i in range(N):
    seed = i * 17 + 3
    _, Q10  = padronizar_curva(*simular_curva_bruta(10,  taxas[i], seed))
    _, Q100 = padronizar_curva(*simular_curva_bruta(100, taxas[i], seed))
    dQ = Q100 - Q10

    ciclos_s = [2, 10, 20, 40, 60, 80, 100]
    caps = [padronizar_curva(*simular_curva_bruta(c, taxas[i], seed))[1].max() for c in ciclos_s]

    features.append([
        np.log10(np.var(dQ)), np.min(dQ), np.mean(dQ),
        np.polyfit(ciclos_s, caps, 1)[0], caps[0], caps[-1]
    ])

X = np.array(features); y = np.log10(vidas)
print(' OK')
print(f'Dataset: {X.shape[0]} celulas x {X.shape[1]} features')
print(f'Vida util: min={vidas.min()}, max={vidas.max()}, media={vidas.mean():.0f} ciclos')


## 2. Por que Regularizacao e Necessaria?

Com **41 celulas de treino** e **6 features** o risco de overfitting e moderado.
Com 20 features (como no artigo) ele seria alto.

**Overfitting:** o modelo memoriza o treino mas falha no teste.
**Solucao:** adicionar uma penalidade matematica aos coeficientes grandes.

Comparamos Minimos Quadrados comum (OLS) vs Elastic Net para ver a diferenca
no gap entre erro de treino e teste.


In [ ]:
idx_tr = np.arange(41); idx_te = np.arange(41, 84)
X_tr, X_te = X[idx_tr], X[idx_te]; y_tr, y_te = y[idx_tr], y[idx_te]

sc = StandardScaler(); X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)

ols = LinearRegression().fit(X_tr_s, y_tr)
en  = ElasticNetCV(l1_ratio=[0.1,0.5,0.7,0.9,0.95,1.0],
                   cv=KFold(n_splits=4, shuffle=True, random_state=0),
                   max_iter=5000).fit(X_tr_s, y_tr)

def rmse_log(modelo, X, y_true):
    return mean_squared_error(y_true, modelo.predict(X), squared=False)

print('Modelo               RMSE Treino  RMSE Teste  Diferenca')
print('----------------------------------------------------------')
for nome, m in [('OLS (sem reg.)', ols), ('Elastic Net', en)]:
    r_tr = rmse_log(m, X_tr_s, y_tr); r_te = rmse_log(m, X_te_s, y_te)
    print(f'{nome:<20} {r_tr:>11.4f}  {r_te:>10.4f}  {r_te-r_tr:>+10.4f}')
print('(RMSE em log10 de ciclos — menor = melhor)')
print()
print(f'Elastic Net selecionado: alpha={en.l1_ratio_:.2f}, lambda={en.alpha_:.4f}')
print('alpha -> mix L1/L2;  lambda -> forca da regularizacao')


## 3. Elastic Net: Combinando L1 (Lasso) e L2 (Ridge)

A Elastic Net minimiza:

    min_w  ||y - Xw||^2  +  lambda * ( alpha * ||w||_1  +  (1-alpha)/2 * ||w||^2_2 )

| Regularizacao | Efeito | Quando usar |
|--------------|--------|-------------|
| **L1 (Lasso)** | Zera coeficientes de features irrelevantes | Selecao automatica de features |
| **L2 (Ridge)** | Encolhe todos os coeficientes | Features correlacionadas entre si |
| **Elastic Net** | Combina os dois | Caso geral — usado no artigo |

O parametro **alpha** controla a mistura (alpha=1 = Lasso, alpha=0 = Ridge).
O parametro **lambda** controla a forca total da penalizacao.
Ambos sao selecionados por **validacao cruzada de 4 folds**.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
cores = ['tomato' if c < 0 else 'steelblue' for c in en.coef_]
ax.bar(nomes, en.coef_, color=cores, edgecolor='gray', lw=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Coeficiente (apos normalizacao)')
ax.set_title('Coeficientes do modelo Elastic Net\n(azul=positivo, vermelho=negativo)')
ax.tick_params(axis='x', rotation=20)

ax = axes[1]
y_pred = en.predict(X_te_s)
vidas_r = 10**y_te; vidas_p = 10**y_pred
ax.scatter(vidas_r, vidas_p, alpha=0.7, color='steelblue', edgecolors='gray', lw=0.5, s=60)
lims = [min(vidas_r.min(),vidas_p.min())*0.9, max(vidas_r.max(),vidas_p.max())*1.1]
ax.plot(lims, lims, 'k--', lw=1.2, label='Predicao perfeita')
ax.set_xlabel('Vida util real (ciclos)'); ax.set_ylabel('Vida util prevista (ciclos)')
ax.set_title('Predito vs Real — conjunto de teste'); ax.legend()

plt.tight_layout(); plt.show()


## 4. Avaliacao: RMSE e Erro Percentual Medio

O artigo reporta duas metricas:

    RMSE = sqrt(mean((y_real - y_pred)^2))         [em ciclos]
    Erro% = mean(|y_real - y_pred| / y_real) * 100

**Benchmark (baseline):** predizer sempre a media do conjunto de treino
dá ~30% de erro. O modelo com apenas 1 feature ja bate esse baseline.


In [ ]:
def metricas(y_r_log, y_p_log, nome):
    yr = 10**y_r_log; yp = 10**y_p_log
    rmse = np.sqrt(mean_squared_error(yr, yp))
    ep   = np.mean(np.abs(yr-yp)/yr)*100
    print(f'  {nome:<28} RMSE = {rmse:>5.0f} ciclos  |  Erro% = {ep:>5.1f}%')
    return rmse, ep

print('Desempenho no TESTE:')
metricas(y_te, en.predict(X_te_s),            'Elastic Net (6 features)')
metricas(y_te, np.full_like(y_te, y_tr.mean()), 'Baseline (media do treino)')
en1 = ElasticNetCV(cv=4, max_iter=5000).fit(X_tr_s[:,[0]], y_tr)
metricas(y_te, en1.predict(X_te_s[:,[0]]),    '1 feature (log_var_dQ)')

y_pred = en.predict(X_te_s)
erros = np.abs(10**y_te - 10**y_pred) / 10**y_te * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
ax.hist(erros, bins=15, color='steelblue', edgecolor='white')
ax.axvline(erros.mean(), color='tomato', lw=2, linestyle='--', label=f'Media = {erros.mean():.1f}%')
ax.set_xlabel('Erro percentual (%)'); ax.set_ylabel('Numero de celulas')
ax.set_title('Distribuicao do erro percentual no teste'); ax.legend()

ax = axes[1]
ordm = np.argsort(10**y_te); xi = np.arange(len(y_te))
ax.scatter(xi, (10**y_te)[ordm],   color='gray',  s=50, label='Real',    zorder=3)
ax.scatter(xi, (10**y_pred)[ordm], color='tomato', s=50, marker='x', label='Previsto', zorder=3)
for x_,yr_,yp_ in zip(xi,(10**y_te)[ordm],(10**y_pred)[ordm]):
    ax.plot([x_,x_],[yr_,yp_],'gray',lw=0.6,alpha=0.5)
ax.set_xlabel('Celulas ordenadas por vida util real'); ax.set_ylabel('Vida util (ciclos)')
ax.set_title('Real vs Previsto'); ax.legend()
plt.tight_layout(); plt.show()


## 5. Classificacao Binaria com os Primeiros 5 Ciclos

Alem da regressao, o artigo treina um **classificador binario** para triagem
na linha de producao: usando apenas os **5 primeiros ciclos**, classifica
cada celula em *vida curta* (<= 550 ciclos) ou *vida longa* (> 550 ciclos).

O modelo: **Regressao Logistica** — aprende uma fronteira de decisao linear.
No artigo com dados reais: **97,5% de acuracia** no teste secundario.

> Por que 5 ciclos? A fabrica pode separar baterias boas das problematicas
> depois de apenas 5 ciclos de teste — economizando semanas de espera.


In [ ]:
LIMIAR = 550
y_clf = (vidas > LIMIAR).astype(int)
X_clf = []
for i in range(N):
    seed = i * 17 + 3
    _, Q4 = padronizar_curva(*simular_curva_bruta(4, taxas[i], seed))
    _, Q5 = padronizar_curva(*simular_curva_bruta(5, taxas[i], seed))
    dQ_c = Q5 - Q4
    X_clf.append([np.log10(np.var(dQ_c)), np.min(dQ_c)])
X_clf = np.array(X_clf)

print(f'Dataset: {X_clf.shape}')
print(f'Classe 0 (vida curta <={LIMIAR}): {(y_clf==0).sum()} celulas')
print(f'Classe 1 (vida longa  >{LIMIAR}): {(y_clf==1).sum()} celulas')

X_c_tr, y_c_tr = X_clf[idx_tr], y_clf[idx_tr]
X_c_te, y_c_te = X_clf[idx_te], y_clf[idx_te]
sc2 = StandardScaler(); X_c_tr_s = sc2.fit_transform(X_c_tr); X_c_te_s = sc2.transform(X_c_te)

clf = LogisticRegression(C=1.0, max_iter=1000).fit(X_c_tr_s, y_c_tr)
acc_tr = accuracy_score(y_c_tr, clf.predict(X_c_tr_s))*100
acc_te = accuracy_score(y_c_te, clf.predict(X_c_te_s))*100
cm = confusion_matrix(y_c_te, clf.predict(X_c_te_s))

print(f'\nAcuracia TREINO: {acc_tr:.1f}%')
print(f'Acuracia TESTE:  {acc_te:.1f}%   (artigo real: 92.7-97.5%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=16, fontweight='bold',
                color='white' if cm[i,j]>cm.max()/2 else 'black')
ax.set_xticks([0,1]); ax.set_xticklabels(['Prev:Curta','Prev:Longa'])
ax.set_yticks([0,1]); ax.set_yticklabels(['Real:Curta','Real:Longa'])
ax.set_title(f'Matriz de Confusao (teste)\nAcuracia = {acc_te:.1f}%')

ax = axes[1]
x0 = np.linspace(X_c_te_s[:,0].min()-0.5, X_c_te_s[:,0].max()+0.5, 300)
x1 = np.linspace(X_c_te_s[:,1].min()-0.5, X_c_te_s[:,1].max()+0.5, 300)
xx0, xx1 = np.meshgrid(x0, x1)
Z = clf.predict(np.c_[xx0.ravel(), xx1.ravel()]).reshape(xx0.shape)
ax.contourf(xx0, xx1, Z, alpha=0.25, cmap='RdYlGn'); ax.contour(xx0, xx1, Z, colors='k', lw=0.8)
sc3 = ax.scatter(X_c_te_s[:,0], X_c_te_s[:,1], c=y_c_te, cmap='RdYlGn',
                 edgecolors='k', lw=0.5, s=60, zorder=3)
ax.set_xlabel('log(Var[DeltaQ_{5-4}]) norm.'); ax.set_ylabel('min(DeltaQ_{5-4}) norm.')
ax.set_title('Fronteira de decisao\nVerde=vida longa, Vermelho=vida curta')
plt.colorbar(sc3, ax=ax, label='Classe real')
plt.tight_layout(); plt.show()


## 6. Resumo Final

| Modelo | Features | Dados | Erro |
|--------|----------|-------|------|
| Elastic Net (1 feature) | log(Var[DeltaQ_{100-10}]) | 100 ciclos | ~11-15% |
| Elastic Net (6-9 features) | Multiplas fontes | 100 ciclos | ~8-11% |
| Classificador Logistico | 18 features | **5 ciclos** | ~5% (erro de classe) |
| Baseline (media) | — | — | ~30-36% |

### Licoes do Artigo

1. **Feature engineering supera dados brutos:** variancia de DeltaQ bate qualquer metrica de capacidade simples
2. **Modelos simples + boas features:** modelo linear supera a literatura sem redes neurais
3. **Interpretabilidade:** cada coeficiente conecta-se ao mecanismo fisico (LAMdeNE, LLI)
4. **Validacao rigorosa:** dataset secundario coletado apos o desenvolvimento confirma a generalizacao

---
*Notebooks baseados em: Severson, Attia et al. — Nature Energy 4, 383-391 (2019)*
